# PGA Tour Exploratory Data Analysis (EDA)
This notebook loads historical PGA Tour results and weather data to run basic analysis and visualizations.
Since we're on the Mac, this is perfect for testing hypotheses before training models on the PC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load the Datasets

In [ ]:
# Load historical results (Sample from 2001-2025)
print("Loading historical results...")
df_results = pd.read_csv('../src/data/archive/pga_results_2001-2025.tsv', sep='\t')
print(f"Results shape: {df_results.shape}")

# Load weather data
print("Loading weather data...")
df_weather = pd.read_csv('../src/data/archive/Tournaments With Weather Data.csv')
print(f"Weather shape: {df_weather.shape}")

display(df_results.head(3))
display(df_weather.head(3))

## 2. Analyzing Scores by Round
Let's see if scoring average drops as the tournament progresses (e.g. Sunday pressure).

In [ ]:
# Convert rounds to numeric, replacing 'WD', 'DQ', etc with NaN
rounds = ['round1', 'round2', 'round3', 'round4']
for r in rounds:
    df_results[r] = pd.to_numeric(df_results[r], errors='coerce')

# Calculate mean score for each round
mean_scores = df_results[rounds].mean()

plt.figure(figsize=(8, 5))
sns.barplot(x=mean_scores.index, y=mean_scores.values, palette="Blues_d")
plt.title('Average Score by Round (Historical PGA Tour)', fontsize=14)
plt.ylabel('Average Score')
plt.ylim(70, 73)  # Zoom in to see the difference
plt.show()

## 3. Top Players' Scoring Distributions
Let's look at the distribution of total scores for a few top players.

In [ ]:
top_players = ['Scottie Scheffler', 'Rory McIlroy', 'Tiger Woods', 'Jon Rahm']

# Filter data for these players
df_top = df_results[df_results['name'].isin(top_players)].copy()
# Convert 'total' to numeric
df_top['total'] = pd.to_numeric(df_top['total'], errors='coerce')
df_top = df_top.dropna(subset=['total'])

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_top, x='total', hue='name', fill=True, common_norm=False, alpha=0.5)
plt.title('Distribution of Total Tournament Scores (Top Players)', fontsize=14)
plt.xlabel('Total Score (Lower is better)')
plt.ylabel('Density')
plt.xlim(260, 300)
plt.show()

## 4. The Effect of Wind on Scoring
We'll join the weather data to see how wind speeds correlate with round 1 scores.

In [ ]:
# Prepare Weather Data
df_weather['season'] = df_weather['year']
df_weather['tournament_lower'] = df_weather['name'].str.lower().str.strip()
df_results['tournament_lower'] = df_results['tournament'].str.lower().str.strip()

# Merge
df_merged = pd.merge(
    df_results, 
    df_weather[['season', 'tournament_lower', 'day0wind', 'day1wind', 'Course Par']], 
    on=['season', 'tournament_lower'], 
    how='inner'
)

# Calculate Score to Par for Round 1
df_merged['r1_to_par'] = df_merged['round1'] - df_merged['Course Par']
df_merged = df_merged.dropna(subset=['r1_to_par', 'day0wind'])

# Plot
plt.figure(figsize=(10, 6))
# We'll take a random sample so the scatter plot isn't too crowded
sample_df = df_merged.sample(n=min(5000, len(df_merged)), random_state=42)

sns.regplot(data=sample_df, x='day0wind', y='r1_to_par', 
            scatter_kws={'alpha':0.1, 'color':'blue'}, line_kws={'color':'red'})
plt.title('Round 1 Score to Par vs. Wind Speed', fontsize=14)
plt.xlabel('Wind Speed (mph)')
plt.ylabel('Score to Par (Round 1)')
plt.show()

# Calculate correlation
corr = sample_df['day0wind'].corr(sample_df['r1_to_par'])
print(f"Correlation between Wind Speed and Score to Par: {corr:.3f}")